# Predicción del amontonamiento de buses en Cusco — Experimentación

Notebook del Entregable 2. Sigue la metodología del Entregable 1:

1. **Preprocesamiento**: pasos por punto de control, limpieza, imputación, headways y clases.
2. **GPS**: detección de anomalías (Isolation Forest), paraderos con DBSCAN y variables por tramo.
3. **Dataset de modelado**: variables conocidas en el instante a(i,k) y objetivo en k+Δ.
4. **Experimentos**: E1 comparación de modelos (Δ=3), E2 horizonte, E3 ablación, E4 PCA,
   E5 hiperparámetros y costo, E6 análisis de errores + prueba de Wilcoxon.

Ejecutar desde la carpeta `TA/` (lee `Dataset/*.csv`). Escribe tablas en `resultados/`,
figuras en `figuras/` y los mejores modelos en `modelos/` (para la demo).
Tiempo aproximado: 15–25 min en una laptop.

In [ ]:
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.cluster import DBSCAN
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import (AdaBoostClassifier, AdaBoostRegressor, HistGradientBoostingClassifier,
                              HistGradientBoostingRegressor, IsolationForest, RandomForestClassifier,
                              RandomForestRegressor)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score,
                             mean_absolute_error, r2_score, recall_score, root_mean_squared_error)
from sklearn.model_selection import ParameterSampler
from sklearn.neighbors import BallTree, KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

SEED = 42
DATA = Path("Dataset")
RES, FIG, MOD = Path("resultados"), Path("figuras"), Path("modelos")
for d in (RES, FIG, MOD):
    d.mkdir(exist_ok=True)

DELTAS = [1, 3, 5]          # horizontes de predicción (puntos de control)
DELTA_MAIN = 3              # horizonte principal (E1)
TRAIN_END, VAL_END = "2025-01-22", "2025-01-26"   # partición temporal por días
MAX_DELAY = 60              # |retraso| > 60 min se considera error de registro
MAX_H = 60                  # headways > 60 min = fin de servicio / error
SLOW = 5                    # km/h: el GPS casi no registra velocidad 0, se usa "detenido o casi detenido"
N_ITER = 6                  # combinaciones de hiperparámetros por modelo
SVM_N = 8000                # submuestra para SVM (costo cuadrático)
CLASSES = ["amontonado", "regular", "hueco"]

## 1. Preprocesamiento de `Times.csv`

Cada fila es una vuelta con tres arreglos de 18 valores (hora programada, hora real y retraso).
Se convierte a una fila por **paso** (vuelta i, punto de control k). Reglas de limpieza:

- `end_time = 00:00:00` → el paso **no fue registrado** (es el origen del `-` en `difference_time`).
- |retraso| > 60 min → error de registro.
- Hora real menor que la de un punto anterior de la misma vuelta → error (no monotónico).
- Isolation Forest sobre el retraso y la desviación del tramo para marcar anomalías restantes.

In [ ]:
t = pd.read_csv(DATA / "Times.csv", dtype={"vehicle_id_id": str})
arr = ["start_time", "end_time", "difference_time"]
for c in arr:
    t[c] = t[c].str.strip("{}").str.split(",")
p = t.explode(arr).reset_index(drop=True)
p["k"] = p.groupby("id").cumcount() + 1
p = p.rename(columns={"id": "trip", "vehicle_id_id": "veh"})
p["s"] = pd.to_timedelta(p.start_time).dt.total_seconds() / 60   # minutos desde medianoche
p["a"] = pd.to_timedelta(p.end_time).dt.total_seconds() / 60
p = p[["trip", "date", "veh", "lap", "k", "s", "a"]].sort_values(["trip", "k"]).reset_index(drop=True)

limpieza = {"pasos totales": len(p)}
p.loc[p.a == 0, "a"] = np.nan
limpieza["no registrados (00:00:00)"] = int(p.a.isna().sum())
p["delay"] = p.a - p.s
bad = p.delay.abs() > MAX_DELAY
limpieza[f"|retraso| > {MAX_DELAY} min"] = int(bad.sum())
p.loc[bad, ["a", "delay"]] = np.nan
prev_max = p.groupby("trip").a.cummax().groupby(p.trip).shift().groupby(p.trip).ffill()
bad = p.a < prev_max
limpieza["no monotónicos"] = int(bad.sum())
p.loc[bad, ["a", "delay"]] = np.nan

# desviación del tramo k-1 -> k respecto a lo programado (= cambio de retraso)
p["ddelay"] = p.delay - p.groupby("trip").delay.shift()
ok = p[["delay", "ddelay"]].notna().all(axis=1)
iso_p = IsolationForest(contamination=0.001, random_state=SEED).fit(p.loc[ok, ["delay", "ddelay"]])
bad = pd.Series(False, index=p.index)
bad[ok] = iso_p.predict(p.loc[ok, ["delay", "ddelay"]]) == -1
limpieza["anomalías Isolation Forest"] = int(bad.sum())
p.loc[bad, ["a", "delay"]] = np.nan
limpieza["pasos válidos"] = int(p.a.notna().sum())
pd.Series(limpieza, name="pasos").to_frame()

**Imputación para el cálculo de headways.** Si al bus de adelante le falta un paso, el headway del
bus de atrás se calcularía contra el bus anterior a ese y parecería un "hueco" falso. Por eso el
retraso faltante se interpola dentro de la misma vuelta (máximo 2 puntos seguidos). Los pasos
imputados **solo** se usan para ordenar buses y calcular headways de otros; nunca como objetivo.

In [ ]:
p["delay_i"] = p.groupby("trip").delay.transform(lambda d: d.interpolate(limit=2, limit_area="inside"))
p["a_i"] = p.s + p.delay_i
p["imp"] = p.a.isna() & p.a_i.notna()
print(f"pasos imputados: {p.imp.sum()}  | sin dato tras imputar: {p.a_i.isna().sum()}")

# headway real con el bus que pasó justo antes por el mismo punto el mismo día
q = p[p.a_i.notna()].sort_values(["date", "k", "a_i"])
g = q.groupby(["date", "k"])
q["trip_j"] = g.trip.shift()
q["h"] = q.a_i - g.a_i.shift()
q["hp"] = q.s - g.s.shift()          # headway programado del mismo par de buses
valid = q.h.between(0, MAX_H) & q.hp.between(0, MAX_H, inclusive="right")
q.loc[~valid, ["h", "hp"]] = np.nan
q["r"] = q.h / q.hp
q["clase"] = np.select([q.r < 0.5, q.r > 1.5], ["amontonado", "hueco"], "regular")
q.loc[q.r.isna(), "clase"] = None
p = p.merge(q[["trip", "k", "trip_j", "h", "hp", "r", "clase"]], on=["trip", "k"], how="left")
print("headways descartados (primer bus del día en cada punto, adelantamientos o > 60 min):", int((~valid).sum()))
p.clase.value_counts(normalize=True).round(3)

### Análisis exploratorio con los datos limpios
Estas cifras y la figura reemplazan a las del Entregable 1, que se calcularon sin limpiar los pasos
no registrados (un paso faltante generaba headways falsos).

In [ ]:
hv = p.dropna(subset=["h"])
cv = hv.h.std() / hv.h.mean()
print(f"headways válidos: {len(hv)} | media {hv.h.mean():.2f} min | CV {cv:.2f} | "
      f"espera esperada {hv.h.mean() / 2 * (1 + cv**2):.2f} min vs {hv.h.mean() / 2:.2f} min si fuera regular "
      f"(+{100 * cv**2:.0f} %)")
by_k = hv.groupby("k").agg(std=("h", "std"), bunch=("clase", lambda c: 100 * (c == "amontonado").mean()))
fig, ax = plt.subplots(figsize=(3.4, 2.0), dpi=300)
ax.plot(by_k.index, by_k["std"], "o-", ms=2.5, color="tab:blue")
ax.set_xlabel("Punto de control k", fontsize=6); ax.set_ylabel("σ del headway (min)", fontsize=6, color="tab:blue")
ax2 = ax.twinx()
ax2.plot(by_k.index, by_k.bunch, "s--", ms=2.5, color="tab:red")
ax2.set_ylabel("% pasos amontonados", fontsize=6, color="tab:red")
for a_ in (ax, ax2):
    a_.tick_params(labelsize=5)
ax.set_xticks(range(1, 19))
fig.tight_layout(); fig.savefig(FIG / "fig2_headway.png"); plt.show()
by_k.round(2).T

## 2. Preprocesamiento de `Monitoring.csv` (GPS)

- Se eliminan registros duplicados (mismo vehículo, fecha y hora).
- **Isolation Forest** sobre (latitud, longitud, velocidad) elimina saltos de GPS y velocidades imposibles.
- **DBSCAN** (distancia haversine, eps = 25 m) sobre los registros detenidos (≤ 5 km/h; el equipo GPS casi
  no registra velocidad 0, solo 0,6 % de los registros) identifica los lugares
  donde los buses se detienen de forma recurrente (paraderos reales, terminal, semáforos).

In [ ]:
m = pd.read_csv(DATA / "Monitoring.csv", dtype={"vehicle_id_id": str},
                usecols=["date", "time", "latitude", "longitude", "speed", "vehicle_id_id"])
m = m.rename(columns={"vehicle_id_id": "veh"})
n0 = len(m)
m = m.drop_duplicates(["veh", "date", "time"])
m["t"] = pd.to_timedelta(m.time).dt.total_seconds() / 60
iso_g = IsolationForest(contamination=0.005, random_state=SEED, n_jobs=-1)
out = iso_g.fit_predict(m[["latitude", "longitude", "speed"]]) == -1
print(f"GPS: {n0} registros, {n0 - len(m)} duplicados, {out.sum()} anomalías (Isolation Forest)")
m = m[~out].reset_index(drop=True)

In [ ]:
EPS = 25 / 6_371_000   # 25 m en radianes
stop = m[m.speed <= SLOW]
samp = stop.sample(min(40_000, len(stop)), random_state=SEED)
Xs = np.radians(samp[["latitude", "longitude"]].to_numpy())
db = DBSCAN(eps=EPS, min_samples=40, metric="haversine", algorithm="ball_tree").fit(Xs)
core, core_lab = Xs[db.core_sample_indices_], db.labels_[db.core_sample_indices_]
# asignar todos los registros detenidos al cluster del punto núcleo más cercano (si está a < eps)
dist, idx = BallTree(core, metric="haversine").query(np.radians(stop[["latitude", "longitude"]].to_numpy()), k=1)
m["par"] = -1
m.loc[stop.index, "par"] = np.where(dist[:, 0] <= EPS, core_lab[idx[:, 0]], -1)
n_par = len(set(core_lab))
print(f"DBSCAN: {n_par} lugares de detención recurrentes; "
      f"{(m.loc[stop.index, 'par'] >= 0).mean():.1%} de los registros detenidos cae en uno")
centros = m[m.par >= 0].groupby("par")[["latitude", "longitude"]].median()
centros["registros"] = m[m.par >= 0].groupby("par").size()
centros.to_csv(RES / "paraderos_dbscan.csv")

fig, ax = plt.subplots(figsize=(7, 3.5))
bg = m.sample(60_000, random_state=SEED)
ax.scatter(bg.longitude, bg.latitude, s=0.2, c="lightgray", rasterized=True)
ax.scatter(centros.longitude, centros.latitude, s=np.clip(centros.registros / 50, 5, 200),
           c="tab:red", alpha=0.6, edgecolor="k", linewidth=0.3)
ax.set(xlabel="Longitud", ylabel="Latitud", title="Lugares de detención recurrentes (DBSCAN)")
ax.set_aspect("equal")
fig.tight_layout(); fig.savefig(FIG / "exp_dbscan_paraderos.png", dpi=200); plt.show()

**Variables GPS por tramo.** Cada registro GPS se asigna al tramo que termina en el siguiente paso
del mismo vehículo (k−1 → k), con `merge_asof`. Por tramo se calcula: velocidad media, fracción de
registros detenidos, fracción detenida en lugares DBSCAN y número de lugares DBSCAN distintos.

In [ ]:
passes = p.loc[p.a_i.notna(), ["veh", "date", "a_i", "trip", "k"]].sort_values("a_i")
mg = pd.merge_asof(m.sort_values("t"), passes, left_on="t", right_on="a_i", by=["veh", "date"],
                   direction="forward", tolerance=60).dropna(subset=["trip"])
mg["stop"] = mg.speed <= SLOW
mg["stop_par"] = mg.stop & (mg.par >= 0)
seg = mg.groupby(["trip", "k"]).agg(v=("speed", "mean"), stop=("stop", "mean"),
                                    stop_par=("stop_par", "mean"),
                                    npar=("par", lambda s: s[s >= 0].nunique())).reset_index()
seg["trip"] = seg.trip.astype(p.trip.dtype)
seg["k"] = seg.k.astype(int)
print(f"tramos con GPS: {len(seg)} de {p.a_i.notna().sum()}")
seg.describe().round(2)

## 3. Dataset de modelado

Para cada paso real (no imputado) del bus i por el punto k, con bus precedente j, se construyen
variables **conocidas en el instante a(i,k)** y el objetivo en k+Δ:

| Grupo | Variables |
|---|---|
| Bus i | h(i,k), hₚ(i,k), r(i,k), δ(i,k), δ(i,k)−δ(i,k−1), h(i,k−1), r(i,k−1), vuelta |
| Bus precedente j | puntos de ventaja, δ y Δδ en su último punto alcanzado, r(j,k) |
| GPS | velocidad y fracción detenida del bus i (tramo k−1→k) y de j (su último tramo) |
| DBSCAN | fracción detenida en paraderos y n.º de paraderos del bus i en su último tramo |
| Contexto | k, hora, día de la semana, hₚ objetivo (programado con j), vehículo |

Objetivo: h(i,k+Δ) (regresión) y su clase (amontonado / regular / hueco).

In [ ]:
key = ["trip", "k"]
p = p.merge(seg, on=key, how="left")
p = p.sort_values(key).reset_index(drop=True)
gt = p.groupby("trip")
p["h_km1"], p["r_km1"] = gt.h.shift(), gt.r.shift()
p["ddelay_i"] = p.delay_i - gt.delay_i.shift()

base = p[p.a.notna() & p.h.notna() & p.trip_j.notna()].copy()
base["trip_j"] = base.trip_j.astype(p.trip.dtype)
base = base.rename(columns={"h": "h_k", "hp": "hp_k", "r": "r_k", "delay": "delay_k", "ddelay_i": "ddelay_k",
                            "v": "v_i", "stop": "stop_i", "stop_par": "stoppar_i", "npar": "npar_i"})
base["hour"] = base.s / 60
base["dow"] = pd.to_datetime(base.date).dt.dayofweek
base["row"] = np.arange(len(base))

# último punto alcanzado por j antes de a(i,k)
jj = base[["row", "trip_j", "a", "k"]].merge(
    p[["trip", "k", "a_i", "delay_i", "ddelay_i", "v", "stop"]].rename(columns={"trip": "trip_j", "k": "kj"}),
    on="trip_j")
jj = jj[jj.a_i <= jj.a].sort_values("kj").groupby("row").tail(1)
jj = jj.assign(lead_j=jj.kj - jj.k).rename(columns={"delay_i": "delay_j", "ddelay_i": "ddelay_j", "v": "v_j", "stop": "stop_j"})
base = base.merge(jj[["row", "lead_j", "delay_j", "ddelay_j", "v_j", "stop_j"]], on="row", how="left")
base = base.merge(p[["trip", "k", "r"]].rename(columns={"trip": "trip_j", "r": "r_j_k"}), on=["trip_j", "k"], how="left")

rows = []
for D in DELTAS:
    b = base[base.k + D <= 18].copy()
    b["delta"], b["kt"] = D, b.k + D
    tgt = p[~p.imp & p.a.notna()][["trip", "k", "h", "clase"]].rename(columns={"k": "kt", "h": "y_h", "clase": "y_c"})
    b = b.merge(tgt, on=["trip", "kt"], how="inner").dropna(subset=["y_h", "y_c"])
    sj = p[["trip", "k", "s"]].rename(columns={"trip": "trip_j", "k": "kt", "s": "s_j"})
    st = p[["trip", "k", "s"]].rename(columns={"k": "kt", "s": "s_t"})
    b = b.merge(sj, on=["trip_j", "kt"], how="left").merge(st, on=["trip", "kt"], how="left")
    b["hp_target"] = b.s_t - b.s_j
    b = b[b.hp_target > 0]   # j programado después de i (adelantamiento previo): sin referencia válida
    rows.append(b)
D_all = pd.concat(rows, ignore_index=True)
D_all["split"] = np.where(D_all.date <= TRAIN_END, "train", np.where(D_all.date <= VAL_END, "val", "test"))

G_I = ["h_k", "hp_k", "r_k", "delay_k", "ddelay_k", "h_km1", "r_km1", "lap"]
G_PRED = ["lead_j", "delay_j", "ddelay_j", "r_j_k"]
G_GPS = ["v_i", "stop_i", "v_j", "stop_j"]
G_DB = ["stoppar_i", "npar_i"]
G_CTX = ["k", "hour", "dow", "hp_target"]
CAT = ["veh"]
NUM = G_I + G_PRED + G_GPS + G_DB + G_CTX
D_all[["date", "split", "delta", "trip", "k", "veh"] + NUM + ["y_h", "y_c"]].to_pickle(RES / "dataset_modelo.pkl")

print(D_all.groupby(["delta", "split"]).size().unstack())
D_all[NUM].describe().T.round(2)

## 4. Modelos, referencias y protocolo

- **Referencias**: *horario* (ĥ = hₚ objetivo, clase "regular") y *persistencia*
  (ĥ = hₚ objetivo · r(i,k), clase actual).
- **Modelos** (scikit-learn): Ridge / logística, KNN, árbol de decisión, SVM-RBF (submuestra),
  Random Forest, AdaBoost, Gradient Boosting (HistGradientBoosting) y MLP.
- **Preprocesamiento**: imputación por mediana + estandarización (numéricas) y one-hot (vehículo).
- **Ajuste**: `N_ITER` combinaciones de la Tabla II evaluadas en validación (MAE / F1 macro);
  el mejor se reentrena con entrenamiento + validación y se evalúa **una vez** en prueba.
- **Desbalance**: pesos de muestra inversos a la frecuencia de clase (excepto KNN y MLP, que no los admiten).

In [ ]:
def prep(num, pca=False):
    steps_num = [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]
    ct = ColumnTransformer([("num", Pipeline(steps_num), num),
                            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])
    return [("prep", ct)] + ([("pca", PCA(n_components=0.95, random_state=SEED))] if pca else [])


REG = {
    "Ridge": (Ridge(), {"alpha": [0.01, 0.1, 1, 10, 100]}),
    "KNN": (KNeighborsRegressor(n_jobs=-1), {"n_neighbors": [5, 15, 51], "weights": ["uniform", "distance"]}),
    "Árbol": (DecisionTreeRegressor(random_state=SEED), {"max_depth": [5, 10, 20], "min_samples_leaf": [1, 20, 100]}),
    "SVM": (SVR(), {"C": [0.1, 1, 10], "gamma": [0.01, 0.1, "scale"]}),
    "RandomForest": (RandomForestRegressor(random_state=SEED, n_jobs=-1),
                     {"n_estimators": [200, 500], "max_depth": [10, 20, None], "min_samples_leaf": [1, 5, 20]}),
    "AdaBoost": (AdaBoostRegressor(DecisionTreeRegressor(max_depth=4), random_state=SEED),
                 {"n_estimators": [100, 300], "learning_rate": [0.1, 1.0]}),
    "GradBoost": (HistGradientBoostingRegressor(random_state=SEED, max_iter=1000, early_stopping=True),
                  {"learning_rate": [0.03, 0.1], "max_depth": [4, 6, 8]}),
    "MLP": (MLPRegressor(random_state=SEED, early_stopping=True, max_iter=300),
            {"hidden_layer_sizes": [(64, 32), (128, 64, 32)], "alpha": [1e-4, 1e-3], "learning_rate_init": [1e-3, 1e-4]}),
}
CLF = {
    "Logística": (LogisticRegression(max_iter=2000), {"C": [0.01, 0.1, 1, 10, 100]}),
    "KNN": (KNeighborsClassifier(n_jobs=-1), {"n_neighbors": [5, 15, 51], "weights": ["uniform", "distance"]}),
    "Árbol": (DecisionTreeClassifier(random_state=SEED), {"max_depth": [5, 10, 20], "min_samples_leaf": [1, 20, 100]}),
    "SVM": (SVC(), {"C": [0.1, 1, 10], "gamma": [0.01, 0.1, "scale"]}),
    "RandomForest": (RandomForestClassifier(random_state=SEED, n_jobs=-1),
                     {"n_estimators": [200, 500], "max_depth": [10, 20, None], "min_samples_leaf": [1, 5, 20]}),
    "AdaBoost": (AdaBoostClassifier(DecisionTreeClassifier(max_depth=4), random_state=SEED),
                 {"n_estimators": [100, 300], "learning_rate": [0.1, 1.0]}),
    "GradBoost": (HistGradientBoostingClassifier(random_state=SEED, max_iter=1000, early_stopping=True),
                  {"learning_rate": [0.03, 0.1], "max_depth": [4, 6, 8]}),
    "MLP": (MLPClassifier(random_state=SEED, early_stopping=True, max_iter=300),
            {"hidden_layer_sizes": [(64, 32), (128, 64, 32)], "alpha": [1e-4, 1e-3], "learning_rate_init": [1e-3, 1e-4]}),
}
NO_SW = {"KNN", "MLP"}


def cls_of_ratio(r):
    return np.select([r < 0.5, r > 1.5], ["amontonado", "hueco"], "regular")


def metrics(task, y, yhat):
    if task == "reg":
        return {"MAE": mean_absolute_error(y, yhat), "RMSE": root_mean_squared_error(y, yhat), "R2": r2_score(y, yhat)}
    return {"F1_macro": f1_score(y, yhat, average="macro"),
            "Recall_amontonado": recall_score(y, yhat, labels=["amontonado"], average="macro"),
            "Accuracy": accuracy_score(y, yhat)}


def baselines(task, d):
    if task == "reg":
        return {"Ref. horario": d.hp_target.to_numpy(), "Ref. persistencia": (d.hp_target * d.r_k).to_numpy()}
    return {"Ref. horario": np.full(len(d), "regular"), "Ref. persistencia": cls_of_ratio(d.r_k.to_numpy())}


def fit(task, name, est, params, d, num, pca=False):
    """Entrena el pipeline con los parámetros dados; SVM usa una submuestra."""
    pipe = Pipeline(prep(num, pca) + [("model", est)]).set_params(**{f"model__{k}": v for k, v in params.items()})
    if name == "SVM" and len(d) > SVM_N:
        d = d.sample(SVM_N, random_state=SEED)
    y = d.y_h if task == "reg" else d.y_c
    kw = {} if task == "reg" or name in NO_SW else {"model__sample_weight": compute_sample_weight("balanced", y)}
    t0 = time.perf_counter()
    pipe.fit(d[num + CAT], y, **kw)
    return pipe, time.perf_counter() - t0


def tune(task, name, d_tr, d_va, num, pca=False):
    est, grid = (REG if task == "reg" else CLF)[name]
    n = int(np.prod([len(v) for v in grid.values()]))
    recs = []
    for prm in ParameterSampler(grid, n_iter=min(N_ITER, n), random_state=SEED):
        pipe, tfit = fit(task, name, est, prm, d_tr, num, pca)
        sc = metrics(task, d_va.y_h if task == "reg" else d_va.y_c, pipe.predict(d_va[num + CAT]))
        recs.append({"modelo": name, **{k: str(v) for k, v in prm.items()}, **sc, "t_fit_s": tfit})
    recs = pd.DataFrame(recs)
    best = recs.sort_values("MAE" if task == "reg" else "F1_macro", ascending=task == "reg").iloc[0]
    best_prm = next(prm for prm in ParameterSampler(grid, n_iter=min(N_ITER, n), random_state=SEED)
                    if all(str(prm[k]) == best[k] for k in prm))
    return best_prm, recs


def evaluate(task, name, prm, d_trva, d_te, num, pca=False):
    est, _ = (REG if task == "reg" else CLF)[name]
    pipe, tfit = fit(task, name, est, prm, d_trva, num, pca)
    t0 = time.perf_counter()
    yhat = pipe.predict(d_te[num + CAT])
    tpred = time.perf_counter() - t0
    y = d_te.y_h if task == "reg" else d_te.y_c
    return pipe, yhat, {**metrics(task, y, yhat), "t_fit_s": tfit, "t_pred_s": tpred}


def split(D):
    d = D_all[D_all.delta == D]
    return d[d.split == "train"], d[d.split == "val"], d[d.split != "test"], d[d.split == "test"]

## E1. Comparación de modelos (Δ = 3)

In [ ]:
d_tr, d_va, d_trva, d_te = split(DELTA_MAIN)
E1, BEST_PRM, SEARCH, PRED = {}, {}, [], {}
for task, models in (("reg", REG), ("clf", CLF)):
    tab = [{"modelo": k, **metrics(task, d_te.y_h if task == "reg" else d_te.y_c, v)}
           for k, v in baselines(task, d_te).items()]
    PRED[task] = dict(baselines(task, d_te))
    for name in models:
        t0 = time.perf_counter()
        prm, recs = tune(task, name, d_tr, d_va, NUM)
        recs["tarea"] = task
        SEARCH.append(recs)
        BEST_PRM[(task, name)] = prm
        pipe, yhat, sc = evaluate(task, name, prm, d_trva, d_te, NUM)
        PRED[task][name] = yhat
        tab.append({"modelo": name, **sc, "mejores_hiperparámetros": str(prm)})
        if name in ("GradBoost", "RandomForest"):
            joblib.dump(pipe, MOD / f"{task}_{name}_delta{DELTA_MAIN}.joblib")
        print(f"[{task}] {name:12s} listo en {time.perf_counter() - t0:5.1f} s -> {sc}")
    E1[task] = pd.DataFrame(tab)
    E1[task].to_csv(RES / f"E1_{task}_delta{DELTA_MAIN}.csv", index=False)
pd.concat(SEARCH).to_csv(RES / "E5_busqueda_hiperparametros.csv", index=False)
E1["reg"].round(3)

In [ ]:
E1["clf"].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (task, col) in zip(axes, (("reg", "MAE"), ("clf", "F1_macro"))):
    e = E1[task].sort_values(col, ascending=task == "reg")
    colors = ["tab:gray" if m.startswith("Ref.") else "tab:blue" for m in e.modelo]
    ax.barh(e.modelo, e[col], color=colors)
    ax.invert_yaxis()
    ax.set_xlabel(col + (" (min)" if col == "MAE" else ""))
    ax.set_title("Regresión del headway" if task == "reg" else "Clasificación del servicio")
    for i, v in enumerate(e[col]):
        ax.text(v, i, f" {v:.2f}", va="center", fontsize=8)
fig.suptitle(f"E1: desempeño en prueba (Δ = {DELTA_MAIN}); gris = referencias")
fig.tight_layout(); fig.savefig(FIG / "exp1_comparacion.png", dpi=200); plt.show()

In [ ]:
best_reg = E1["reg"][~E1["reg"].modelo.str.startswith("Ref.")].sort_values("MAE").modelo.iloc[0]
best_clf = E1["clf"][~E1["clf"].modelo.str.startswith("Ref.")].sort_values("F1_macro", ascending=False).modelo.iloc[0]
print("mejor regresor:", best_reg, "| mejor clasificador:", best_clf)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(d_te.y_c, PRED["clf"][best_clf], labels=CLASSES, normalize="true",
                                        ax=axes[0], colorbar=False, values_format=".2f")
axes[0].set_title(f"{best_clf}: matriz de confusión (normalizada por fila)")
axes[1].hexbin(d_te.y_h, PRED["reg"][best_reg], gridsize=50, bins="log", cmap="Blues", extent=(0, 30, 0, 30))
axes[1].plot([0, 30], [0, 30], "r--", lw=1)
axes[1].set(xlabel="headway real (min)", ylabel="headway predicho (min)", title=f"{best_reg}: real vs. predicho")
fig.tight_layout(); fig.savefig(FIG / "exp1_confusion_dispersion.png", dpi=200); plt.show()

## E2. Efecto del horizonte de predicción Δ
Se reutilizan los mejores hiperparámetros de Δ = 3 para todos los modelos.

In [ ]:
E2 = []
for D in DELTAS:
    _, _, tv, te = split(D)
    for task, models in (("reg", REG), ("clf", CLF)):
        y = te.y_h if task == "reg" else te.y_c
        for k, v in baselines(task, te).items():
            E2.append({"delta": D, "tarea": task, "modelo": k, **metrics(task, y, v)})
        for name in models:
            if D == DELTA_MAIN:
                sc = E1[task].set_index("modelo").loc[name].to_dict()
            else:
                sc = evaluate(task, name, BEST_PRM[(task, name)], tv, te, NUM)[2]
            E2.append({"delta": D, "tarea": task, "modelo": name, **sc})
E2 = pd.DataFrame(E2)
E2.to_csv(RES / "E2_horizonte.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (task, col) in zip(axes, (("reg", "MAE"), ("clf", "F1_macro"))):
    for name, g in E2[E2.tarea == task].groupby("modelo"):
        ax.plot(g.delta, g[col], "o--" if name.startswith("Ref.") else "o-", label=name)
    ax.set(xlabel="Δ (puntos de control de anticipación)", ylabel=col, xticks=DELTAS)
axes[1].legend(fontsize=7, ncol=2)
fig.suptitle("E2: desempeño según el horizonte de predicción")
fig.tight_layout(); fig.savefig(FIG / "exp2_horizonte.png", dpi=200); plt.show()
E2.pivot_table(index="modelo", columns=["tarea", "delta"], values=["MAE", "F1_macro"]).round(3).dropna(axis=1, how="all")

## E3. Ablación de grupos de variables (Gradient Boosting, Δ = 3)

In [ ]:
ABL = {"todas": NUM, "sin GPS": [c for c in NUM if c not in G_GPS],
       "sin bus precedente": [c for c in NUM if c not in G_PRED],
       "sin DBSCAN": [c for c in NUM if c not in G_DB],
       "solo bus i + contexto": G_I + G_CTX}
E3 = []
for task in ("reg", "clf"):
    for lab, cols in ABL.items():
        sc = evaluate(task, "GradBoost", BEST_PRM[(task, "GradBoost")], d_trva, d_te, cols)[2]
        E3.append({"tarea": task, "variables": lab, "n_variables": len(cols), **sc})
E3 = pd.DataFrame(E3)
E3.to_csv(RES / "E3_ablacion.csv", index=False)
E3.round(3)

## E4. Reducción de dimensionalidad con PCA (KNN y SVM, Δ = 3)
PCA conserva el 95 % de la varianza después de estandarizar y aplicar one-hot.

In [ ]:
E4 = []
for task in ("reg", "clf"):
    for name in ("KNN", "SVM"):
        for pca in (False, True):
            pipe, _, sc = evaluate(task, name, BEST_PRM[(task, name)], d_trva, d_te, NUM, pca=pca)
            ncomp = pipe.named_steps["pca"].n_components_ if pca else len(pipe.named_steps["prep"].get_feature_names_out())
            E4.append({"tarea": task, "modelo": name, "PCA": pca, "dimensiones": ncomp, **sc})
E4 = pd.DataFrame(E4)
E4.to_csv(RES / "E4_pca.csv", index=False)
E4.round(3)

## E5. Sensibilidad a hiperparámetros y costo computacional
Cada fila de la búsqueda es una combinación evaluada en validación (guardada en
`resultados/E5_busqueda_hiperparametros.csv`). Se resume la dispersión del puntaje por modelo
(cuánto importa ajustar) y los tiempos de entrenamiento y predicción del modelo final.

In [ ]:
S = pd.concat(SEARCH)
E5 = pd.concat([
    S[S.tarea == "reg"].groupby("modelo").MAE.agg(["min", "max", "std"]).add_prefix("MAE_val_"),
    S[S.tarea == "clf"].groupby("modelo").F1_macro.agg(["max", "min", "std"]).add_prefix("F1_val_"),
    E1["reg"].set_index("modelo")[["t_fit_s", "t_pred_s"]].add_prefix("reg_"),
    E1["clf"].set_index("modelo")[["t_fit_s", "t_pred_s"]].add_prefix("clf_"),
], axis=1).dropna(how="all")
E5.to_csv(RES / "E5_resumen.csv")
E5.round(3)

## E6. Análisis de errores y prueba estadística
Error absoluto del mejor regresor frente a la referencia de persistencia, por punto de control y hora.
La prueba de Wilcoxon compara el MAE por bloques (día × punto de control) del conjunto de prueba.

In [ ]:
err = d_te[["date", "k", "hour"]].copy()
err["modelo"] = np.abs(d_te.y_h.to_numpy() - PRED["reg"][best_reg])
err["persistencia"] = np.abs(d_te.y_h.to_numpy() - PRED["reg"]["Ref. persistencia"])
err["horario"] = np.abs(d_te.y_h.to_numpy() - PRED["reg"]["Ref. horario"])
err["hora"] = err.hour.astype(int)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col, lab in ((axes[0], "k", "Punto de control de origen k"), (axes[1], "hora", "Hora del día")):
    e = err.groupby(col)[["modelo", "persistencia", "horario"]].mean()
    e.plot(ax=ax, marker="o", ms=3)
    ax.set(xlabel=lab, ylabel="MAE (min)")
axes[0].legend([best_reg, "Ref. persistencia", "Ref. horario"], fontsize=8)
axes[1].get_legend().remove()
fig.suptitle(f"E6: error de predicción del headway (Δ = {DELTA_MAIN}, prueba)")
fig.tight_layout(); fig.savefig(FIG / "exp6_errores.png", dpi=200); plt.show()

blk = err.groupby(["date", "k"])[["modelo", "persistencia", "horario"]].mean()
W = []
for ref in ("persistencia", "horario"):
    stat, pval = wilcoxon(blk.modelo, blk[ref], alternative="less")
    W.append({"comparación": f"{best_reg} vs. {ref}", "bloques": len(blk), "MAE_modelo": blk.modelo.mean(),
              "MAE_ref": blk[ref].mean(), "mejora_%": 100 * (1 - blk.modelo.mean() / blk[ref].mean()), "p_valor": pval})
W = pd.DataFrame(W)
W.to_csv(RES / "E6_wilcoxon.csv", index=False)
W.round(4)

## Resumen para el informe
La hipótesis del Entregable 1 plantea que el mejor modelo reduce el MAE en al menos 20 % frente a
la persistencia. La celda siguiente imprime los números clave para el paper.

In [ ]:
mae_p = E1["reg"].set_index("modelo").loc["Ref. persistencia", "MAE"]
mae_b = E1["reg"].set_index("modelo").loc[best_reg, "MAE"]
print(f"Δ={DELTA_MAIN} | mejor regresor {best_reg}: MAE {mae_b:.2f} min vs persistencia {mae_p:.2f} "
      f"({100 * (1 - mae_b / mae_p):.1f} % de mejora) -> hipótesis {'confirmada' if mae_b <= 0.8 * mae_p else 'no confirmada'}")
f1_b = E1["clf"].set_index("modelo").loc[best_clf]
print(f"mejor clasificador {best_clf}: F1 macro {f1_b.F1_macro:.3f}, recall amontonado {f1_b.Recall_amontonado:.3f}")
print("archivos:", sorted(str(x) for x in list(RES.iterdir()) + list(MOD.iterdir())))